# Citadel Budgets - Validation Prerequisites Checker

**Purpose:** Verify that all required Azure resources, permissions, and configurations are in place before running the main validation notebooks.

**Run this first** before executing:
- `citadel-anthropic-surface-tests.ipynb`
- `citadel-budget-enforcement-tests.ipynb`

## Prerequisites

1. Azure CLI authenticated (`az login`)
2. Python packages: `azure-identity`, `azure-mgmt-resource`, `azure-mgmt-cosmosdb`, `azure-mgmt-apimanagement`
3. Environment variables (or update cells below):
   - `RESOURCE_GROUP`
   - `SUBSCRIPTION_ID`

In [ ]:
import os
import sys
from azure.identity import DefaultAzureCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cosmosdb import CosmosDBManagementClient
from azure.mgmt.apimanagement import ApiManagementClient
from azure.mgmt.web import WebSiteManagementClient
import requests

# Configuration
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "<your-subscription-id>")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP", "citadel-budgets-dev")

# Expected resource names (update based on your deployment)
EXPECTED_RESOURCES = {
    "cosmos_account": "citadel-budgets-dev-cosmos",
    "apim_name": "citadel-budgets-dev-apim",
    "function_app": "citadel-tier-sync-dev",
    "log_analytics_workspace": "citadel-budgets-dev-logs"
}

# Authenticate
credential = DefaultAzureCredential()

print("✅ Azure credentials loaded")
print(f"Subscription: {SUBSCRIPTION_ID}")
print(f"Resource Group: {RESOURCE_GROUP}")

## 1. Check Resource Group Exists

In [ ]:
resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)

try:
    rg = resource_client.resource_groups.get(RESOURCE_GROUP)
    print(f"✅ Resource group '{RESOURCE_GROUP}' exists in {rg.location}")
except Exception as e:
    print(f"❌ Resource group '{RESOURCE_GROUP}' not found")
    print(f"   Create it with: az group create -n {RESOURCE_GROUP} -l eastus")
    sys.exit(1)

## 2. Check Cosmos DB Account & Containers

In [ ]:
cosmos_client = CosmosDBManagementClient(credential, SUBSCRIPTION_ID)

cosmos_account_name = EXPECTED_RESOURCES["cosmos_account"]
required_containers = ["budgets", "user-tier", "ai-usage-monthly"]

try:
    cosmos_account = cosmos_client.database_accounts.get(RESOURCE_GROUP, cosmos_account_name)
    print(f"✅ Cosmos DB account '{cosmos_account_name}' exists")
    
    # Check database
    db_name = "ai-usage-db"
    try:
        database = cosmos_client.sql_resources.get_sql_database(RESOURCE_GROUP, cosmos_account_name, db_name)
        print(f"✅ Database '{db_name}' exists")
        
        # Check containers
        containers = cosmos_client.sql_resources.list_sql_containers(RESOURCE_GROUP, cosmos_account_name, db_name)
        container_names = [c.name for c in containers]
        
        for required_container in required_containers:
            if required_container in container_names:
                print(f"✅ Container '{required_container}' exists")
            else:
                print(f"❌ Container '{required_container}' NOT FOUND")
                print(f"   Deploy with: az deployment group create -f bicep/infra/main.citadel.bicep")
    except Exception as e:
        print(f"❌ Database '{db_name}' not found: {e}")
except Exception as e:
    print(f"❌ Cosmos DB account '{cosmos_account_name}' not found: {e}")

## 3. Check APIM Instance & APIs

In [ ]:
apim_client = ApiManagementClient(credential, SUBSCRIPTION_ID)
apim_name = EXPECTED_RESOURCES["apim_name"]

try:
    apim = apim_client.api_management_service.get(RESOURCE_GROUP, apim_name)
    print(f"✅ APIM instance '{apim_name}' exists")
    print(f"   Gateway URL: {apim.gateway_url}")
    
    # Check for Anthropic API
    apis = apim_client.api.list_by_service(RESOURCE_GROUP, apim_name)
    api_names = [api.name for api in apis]
    
    if "anthropic-api" in api_names:
        print(f"✅ Anthropic API exists in APIM")
    else:
        print(f"⚠️  Anthropic API not found - it may use a different name")
        print(f"   Available APIs: {', '.join(api_names[:5])}...")
    
    # Check managed identity
    if apim.identity and apim.identity.type == "SystemAssigned":
        print(f"✅ APIM has system-assigned managed identity")
        print(f"   Principal ID: {apim.identity.principal_id}")
    else:
        print(f"❌ APIM does NOT have managed identity enabled")
        print(f"   Enable with: az apim update -n {apim_name} -g {RESOURCE_GROUP} --set identity.type=SystemAssigned")
except Exception as e:
    print(f"❌ APIM instance '{apim_name}' not found: {e}")

## 4. Check tier-sync Function App

In [ ]:
web_client = WebSiteManagementClient(credential, SUBSCRIPTION_ID)
function_app_name = EXPECTED_RESOURCES["function_app"]

try:
    function_app = web_client.web_apps.get(RESOURCE_GROUP, function_app_name)
    print(f"✅ Function App '{function_app_name}' exists")
    
    # Check managed identity
    if function_app.identity and function_app.identity.type == "SystemAssigned":
        print(f"✅ Function has system-assigned managed identity")
        print(f"   Principal ID: {function_app.identity.principal_id}")
    else:
        print(f"❌ Function does NOT have managed identity enabled")
    
    # Check application settings
    app_settings = web_client.web_apps.list_application_settings(RESOURCE_GROUP, function_app_name)
    required_settings = ["COSMOS_ACCOUNT_NAME", "TIER_GROUP_OID_SILVER", "TIER_GROUP_OID_GOLD"]
    
    for setting in required_settings:
        if setting in app_settings.properties:
            value = app_settings.properties[setting]
            if value.startswith("<"):
                print(f"⚠️  Setting '{setting}' is still a placeholder: {value}")
            else:
                print(f"✅ Setting '{setting}' is configured")
        else:
            print(f"❌ Setting '{setting}' is MISSING")
except Exception as e:
    print(f"❌ Function App '{function_app_name}' not found: {e}")

## 5. Check RBAC Permissions

In [ ]:
from azure.mgmt.authorization import AuthorizationManagementClient

auth_client = AuthorizationManagementClient(credential, SUBSCRIPTION_ID)

print("\n🔐 Checking RBAC role assignments...")

# Get Cosmos account resource ID
cosmos_resource_id = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.DocumentDB/databaseAccounts/{cosmos_account_name}"

try:
    # List role assignments on Cosmos account
    assignments = auth_client.role_assignments.list_for_resource(
        RESOURCE_GROUP,
        "Microsoft.DocumentDB",
        "",
        "databaseAccounts",
        cosmos_account_name
    )
    
    apim_principal_id = apim.identity.principal_id if apim and apim.identity else None
    function_principal_id = function_app.identity.principal_id if function_app and function_app.identity else None
    
    apim_has_cosmos_role = False
    function_has_cosmos_role = False
    
    for assignment in assignments:
        if assignment.principal_id == apim_principal_id:
            apim_has_cosmos_role = True
            print(f"✅ APIM has role assignment on Cosmos")
        if assignment.principal_id == function_principal_id:
            function_has_cosmos_role = True
            print(f"✅ tier-sync Function has role assignment on Cosmos")
    
    if not apim_has_cosmos_role:
        print(f"❌ APIM does NOT have Cosmos Data Contributor role")
        print(f"   See docs/security-hardening.md section 1")
    
    if not function_has_cosmos_role:
        print(f"❌ tier-sync Function does NOT have Cosmos Data Contributor role")
        print(f"   See docs/security-hardening.md section 1")
except Exception as e:
    print(f"⚠️  Could not check RBAC assignments: {e}")
    print(f"   You may need higher permissions to list role assignments")

## 6. Check Graph API Permissions (tier-sync Function)

In [ ]:
print("\n📊 Checking Microsoft Graph permissions...")
print("⚠️  This requires Global Administrator or Privileged Role Administrator")

try:
    # Get access token for Graph
    token = credential.get_token("https://graph.microsoft.com/.default")
    headers = {"Authorization": f"Bearer {token.token}"}
    
    # Get service principal for tier-sync Function
    function_principal_id = function_app.identity.principal_id
    sp_url = f"https://graph.microsoft.com/v1.0/servicePrincipals/{function_principal_id}/appRoleAssignments"
    
    response = requests.get(sp_url, headers=headers)
    
    if response.status_code == 200:
        assignments = response.json().get("value", [])
        
        required_permissions = {
            "5b567255-7703-4780-807c-7be8301ae99b": "Group.Read.All",
            "7ab1d382-f21e-4acd-a863-ba3e13f7da61": "Directory.Read.All"
        }
        
        granted_permissions = [a["appRoleId"] for a in assignments]
        
        for perm_id, perm_name in required_permissions.items():
            if perm_id in granted_permissions:
                print(f"✅ tier-sync Function has '{perm_name}' permission")
            else:
                print(f"❌ tier-sync Function MISSING '{perm_name}' permission")
                print(f"   See docs/security-hardening.md section 1")
    else:
        print(f"⚠️  Could not retrieve Graph permissions (HTTP {response.status_code})")
        print(f"   You may need Global Administrator role to view app role assignments")
except Exception as e:
    print(f"⚠️  Error checking Graph permissions: {e}")

## 7. Summary & Next Steps

In [ ]:
print("\n" + "="*60)
print("VALIDATION PREREQUISITES SUMMARY")
print("="*60)
print("\nIf all checks above show ✅, you're ready to run:")
print("  - validation/citadel-anthropic-surface-tests.ipynb")
print("  - validation/citadel-budget-enforcement-tests.ipynb")
print("\nIf any checks show ❌, fix them before proceeding:")
print("  - Deploy infrastructure: az deployment group create -f bicep/infra/main.citadel.bicep")
print("  - Configure RBAC: see docs/security-hardening.md")
print("  - Update app settings: see tier-sync-function README.md")
print("\n" + "="*60)